<a href="https://colab.research.google.com/github/DQN-Labs/Chess_AI/blob/main/llama_cpp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title #Runtime Info
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)
from psutil import virtual_memory
ram_gb = virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))
if ram_gb < 20:
  print('Not using a high-RAM runtime')
else:
  print('You are using a high-RAM runtime!')


In [ ]:
#@title ⚡ DQN Labs Ultra-Fast Setup (dqnGPT)

%cd /content/

# Clean previous stuff
!rm -rf build llama_bin.tar.gz *.gguf
print("removed old build")

# 1. Download prebuilt llama.cpp CUDA runtime
!wget -q https://huggingface.co/DQN-Labs/llamacpp-binaries-for-colab/resolve/main/llama_bin.tar.gz
print("got new build")
# Extract runtime
!tar -xzf llama_bin.tar.gz
print("extracted new build")
# 2. Download YOUR model
!wget -q https://huggingface.co/DQN-Labs/dqnGPT-v0.1-3.8B/resolve/main/dqnGPT-v0.1-3.8B.Q4_K_M.gguf
print("got gguf")
# 3. Verify
!ls build/bin | head -5

In [ ]:
!wget https://huggingface.co/DQN-Labs/dqnGPT-v0.1-3.8B/resolve/main/phi-3-mini-4k-instruct.Q4_K_M.gguf

In [ ]:
!export LD_LIBRARY_PATH=/content/build/bin:$LD_LIBRARY_PATH && \
./build/bin/llama-server \
    -m phi-3-mini-4k-instruct.Q4_K_M.gguf \
    -ngl 999 \
    -c 4096 \
    --port 8000

In [ ]:
!apt-get install cloudflared

In [ ]:
from fastapi import FastAPI
import requests
import subprocess
import time

app = FastAPI()

COLAB_URL = "http://YOUR_COLAB_IP:8000"
MODEL_PATH = "/content/dqnGPT.gguf"

server_process = None
model_loaded = False


def start_model():
    global server_process, model_loaded

    if model_loaded:
        return

    print("Starting model...")

    server_process = subprocess.Popen([
        "./build/bin/llama-server",
        "-m", MODEL_PATH,
        "-ngl", "999",
        "-c", "4096",
        "--port", "8000"
    ])

    time.sleep(8)  # wait for startup
    model_loaded = True


@app.get("/v1/models")
def models():
    return {
        "data": [
            {"id": "dqngpt", "object": "model"}
        ]
    }


@app.post("/v1/chat/completions")
def chat(req: dict):
    start_model()

    messages = req["messages"]

    # Convert → llama format
    prompt = ""
    for msg in messages:
        if msg["role"] == "user":
            prompt += f"<|user|>\n{msg['content']}<|end|>\n"
        elif msg["role"] == "assistant":
            prompt += f"<|assistant|>\n{msg['content']}<|end|>\n"

    prompt += "<|assistant|>\n"

    # Call llama.cpp
    res = requests.post(
        f"{COLAB_URL}/completion",
        json={
            "prompt": prompt,
            "n_predict": req.get("max_tokens", 200),
            "temperature": req.get("temperature", 0.7)
        }
    ).json()

    text = res["content"].strip('"')

    return {
        "id": "chatcmpl-123",
        "object": "chat.completion",
        "choices": [
            {
                "index": 0,
                "message": {
                    "role": "assistant",
                    "content": text
                },
                "finish_reason": "stop"
            }
        ],
        "usage": {
            "prompt_tokens": res["tokens_evaluated"],
            "completion_tokens": res["tokens_predicted"],
            "total_tokens": res["tokens_evaluated"] + res["tokens_predicted"]
        }
    }

In [ ]:
!wget -q https://huggingface.co/api/models/DQN-Labs/dqnGPT-v0.1-3.8B -O model_info.json
!cat model_info.json | grep gguf

In [ ]:
!find /content -name "llama-server"

In [ ]:
#@title # Connect using the following address when the server is up.
from google.colab.output import eval_js
print(eval_js("google.colab.kernel.proxyPort(8000)"))

In [ ]:
#@title # inference
%cd /content/llama.cpp
!./server -m zephyr-7b-beta.Q6_K.gguf -ngl 9999 -c 0 --port 12345
